# 第 4 周练习 —— Ruby on Rails 代码 RSpec 生成器

## 练习目标

上传一份 **zip** 打包的 Ruby / Rails 代码库，用 **OpenAI** 为其中每个 `.rb` 文件生成 **RSpec** 测试，再用 Gradio 提供「上传 → 下载」界面。

适合：手里有业务代码、但测试覆盖不足，想快速起草测试草稿时使用。

## 和本课第 4 周的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions | `client.chat.completions.create` |
| Gradio 文件 I/O | `gr.File` 上传 zip / 下载结果 zip |
| 批处理代码助手 | 遍历解压目录，对每个 `.rb` 调一次模型 |

## 怎么跑

1. `.env` 里准备好 `OPENAI_API_KEY`
2. 依次运行导入与 `procesar_archivos` 定义格
3. `demo.launch()` 后上传 zip，下载 `generated_rspec.zip`


In [ ]:
# ========== 导入 + 加载环境变量 + 校验 OpenAI Key ==========

# os：读环境变量
import os
# load_dotenv：把 .env 密钥读进进程环境
from dotenv import load_dotenv
# OpenAI 官方客户端
from openai import OpenAI
# Gradio：后面搭上传/下载界面
import gradio as gr
# zipfile：解压上传包 / 打包结果
import zipfile
# tempfile：临时目录，避免弄脏工作区
import tempfile
# Path：路径工具（本格导入备用）
from pathlib import Path

# override=True：.env 中的值覆盖已有环境变量
load_dotenv(override=True)
# 读取 OpenAI API Key
api_key = os.getenv('OPENAI_API_KEY')

# 粗检：以 sk-proj- 开头且长度 > 10 视为「看起来已配置」
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("OpenAI API Key was correctly setup.")
else:
    # 提示文案保持原样（含原文 OpenAPI 拼写）
    print("OpenAPI API Key is missing.")
# 创建客户端；默认从环境变量 OPENAI_API_KEY 取密钥
client = OpenAI()


In [ ]:
def procesar_archivos(input_zip):
    """Gradio 回调：解压上传 zip → 对每个 .rb 生成 RSpec → 打成可下载 zip。"""
    # TemporaryDirectory：退出 with 后自动删除临时文件
    with tempfile.TemporaryDirectory() as tmpdir:
        # 结果 zip / 解压目录 / 处理后文件目录
        output_path = os.path.join(tmpdir, "resultado.zip")
        extracted_path = os.path.join(tmpdir, "extraidos")
        processed_path = os.path.join(tmpdir, "procesados")
        # 确保输出目录存在
        os.makedirs(processed_path, exist_ok=True)

        # 1) 解压用户上传的 zip（Gradio File 对象用 .name 取本地临时路径）
        with zipfile.ZipFile(input_zip.name, 'r') as zip_ref:
            zip_ref.extractall(extracted_path)

        # 2) 递归走解压树，只处理 .rb 源文件
        for root, _, files in os.walk(extracted_path):
            for file in files:
                if file.endswith(".rb"):
                    # 拼出源文件绝对路径并读入文本
                    input_file_path = os.path.join(root, file)
                    with open(input_file_path, 'r', encoding='utf-8') as f:
                        contenido = f.read()

                    # 调 Chat Completions：请模型为这段 Ruby 生成 RSpec
                    # prompt / model id 必须保持原样
                    response = client.chat.completions.create(
                        model="gpt-4.1-mini",
                        messages=[{"role": "user", "content": f"Generate RSpec tests for this code: {contenido}"}]
                    )
                    
                    # 取出模型返回的测试代码文本
                    rspec_code = response.choices[0].message.content
                    # 输出文件名：原名去扩展名 + _rspec.rb
                    filename = os.path.splitext(file)[0] + "_rspec.rb"
                    with open(os.path.join(processed_path, f"{filename}"), "w") as f:
                        f.write(rspec_code)

        # 3) 把 processed_path 里所有生成文件打进结果 zip
        with zipfile.ZipFile(output_path, 'w') as zip_out:
            for root, _, files in os.walk(processed_path):
                for file in files:
                    # arcname=file：zip 内只用文件名，不保留目录层级
                    zip_out.write(os.path.join(root, file), file)

        # Gradio 下载需要进程退出后仍存在的路径；拷到当前工作目录
        final_zip = os.path.join(os.getcwd(), "generated_rspec.zip")
        import shutil
        shutil.copy(output_path, final_zip)
        
        # 返回路径给 gr.File 输出组件
        return final_zip


In [ ]:
# ========== Gradio Interface：上传 Rails zip → 下载 RSpec zip ==========

# Interface：单函数入出最简单的 UI 形态
demo = gr.Interface(
    # 处理函数：上一格定义的 procesar_archivos
    fn=procesar_archivos,
    # 输入：只接受 .zip
    inputs=gr.File(label="Upload your Ruby on Rails codebase", file_types=[".zip"]),
    # 输出：可下载的结果文件
    outputs=gr.File(label="Download your RSpec tests"),
    # 标题与说明（UI 字符串保持原样）
    title="RSPEC generator with OpenAI",
    description="Upload your Ruby on Rails codebase to generate RSpec tests."
)

# 脚本直接运行时才 launch；被 import 时不自动弹窗
if __name__ == "__main__":
    demo.launch()
